# Google Ads Analysis

This notebook connects to your Google Ads account and provides basic analysis of your campaigns.

**Before running:** Make sure your `.env` file has all credentials filled in.

In [1]:
# Setup - Run this cell first
import pandas as pd
import matplotlib.pyplot as plt
from config import get_google_ads_client, get_customer_id

client = get_google_ads_client()
customer_id = get_customer_id()
ga_service = client.get_service("GoogleAdsService")

print(f"Connected to Google Ads account: {customer_id}")

Matplotlib is building the font cache; this may take a moment.
/Users/sakshibatavia/google-ads-analysis/venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/sakshibatavia/google-ads-analysis/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sakshibatavia/google-ads-analysis/venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug

Connected to Google Ads account: 8915520607


## 1. Campaign Performance Overview
See how all your campaigns are performing (last 30 days).

In [2]:
# Fetch campaign performance data (last 30 days)
query = """
    SELECT
        campaign.name,
        campaign.status,
        metrics.impressions,
        metrics.clicks,
        metrics.ctr,
        metrics.average_cpc,
        metrics.cost_micros,
        metrics.conversions,
        metrics.cost_per_conversion
    FROM campaign
    WHERE segments.date DURING LAST_30_DAYS
        AND campaign.status != 'REMOVED'
    ORDER BY metrics.cost_micros DESC
"""

response = ga_service.search(customer_id=customer_id, query=query)

campaigns = []
for row in response:
    campaigns.append({
        "Campaign": row.campaign.name,
        "Status": row.campaign.status.name,
        "Impressions": row.metrics.impressions,
        "Clicks": row.metrics.clicks,
        "CTR": f"{row.metrics.ctr:.2%}",
        "Avg CPC": f"${row.metrics.average_cpc / 1_000_000:.2f}",
        "Cost": f"${row.metrics.cost_micros / 1_000_000:.2f}",
        "Conversions": row.metrics.conversions,
        "Cost/Conv": f"${row.metrics.cost_per_conversion / 1_000_000:.2f}" if row.metrics.conversions > 0 else "N/A",
    })

df_campaigns = pd.DataFrame(campaigns)
print(f"Found {len(df_campaigns)} campaigns\n")
df_campaigns

Request made: ClientCustomerId: 8915520607, Host: googleads.googleapis.com, Method: /google.ads.googleads.v18.services.GoogleAdsService/Search, RequestId: None, IsFault: True, FaultMessage: GRPC target method can't be resolved.


MethodNotImplemented: 501 GRPC target method can't be resolved.

## 2. Campaign Spend Chart
Visual breakdown of spend by campaign.

In [ ]:
# Bar chart of spend by campaign
if not df_campaigns.empty:
    cost_data = df_campaigns.copy()
    cost_data["Cost_numeric"] = cost_data["Cost"].str.replace("$", "", regex=False).astype(float)
    cost_data = cost_data.sort_values("Cost_numeric", ascending=True)

    fig, ax = plt.subplots(figsize=(10, max(4, len(cost_data) * 0.5)))
    ax.barh(cost_data["Campaign"], cost_data["Cost_numeric"], color="#4285F4")
    ax.set_xlabel("Cost ($)")
    ax.set_title("Spend by Campaign (Last 30 Days)")
    plt.tight_layout()
    plt.show()
else:
    print("No campaign data to chart.")

## 3. Daily Spend Trend
See how your total spend trends day-by-day.

In [ ]:
# Daily spend trend (last 30 days)
query_daily = """
    SELECT
        segments.date,
        metrics.cost_micros,
        metrics.clicks,
        metrics.impressions,
        metrics.conversions
    FROM campaign
    WHERE segments.date DURING LAST_30_DAYS
        AND campaign.status != 'REMOVED'
    ORDER BY segments.date ASC
"""

response_daily = ga_service.search(customer_id=customer_id, query=query_daily)

daily_data = {}
for row in response_daily:
    date = row.segments.date
    if date not in daily_data:
        daily_data[date] = {"Cost": 0, "Clicks": 0, "Impressions": 0, "Conversions": 0}
    daily_data[date]["Cost"] += row.metrics.cost_micros / 1_000_000
    daily_data[date]["Clicks"] += row.metrics.clicks
    daily_data[date]["Impressions"] += row.metrics.impressions
    daily_data[date]["Conversions"] += row.metrics.conversions

df_daily = pd.DataFrame.from_dict(daily_data, orient="index")
df_daily.index = pd.to_datetime(df_daily.index)
df_daily = df_daily.sort_index()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(df_daily.index, df_daily["Cost"], color="#EA4335", linewidth=2)
axes[0].set_ylabel("Cost ($)")
axes[0].set_title("Daily Spend Trend")
axes[0].fill_between(df_daily.index, df_daily["Cost"], alpha=0.1, color="#EA4335")

axes[1].plot(df_daily.index, df_daily["Clicks"], color="#4285F4", linewidth=2, label="Clicks")
axes[1].set_ylabel("Clicks")
axes[1].set_title("Daily Clicks")
axes[1].fill_between(df_daily.index, df_daily["Clicks"], alpha=0.1, color="#4285F4")

plt.tight_layout()
plt.show()

print(f"\nTotal spend (30 days): ${df_daily['Cost'].sum():.2f}")
print(f"Average daily spend: ${df_daily['Cost'].mean():.2f}")
print(f"Total clicks: {int(df_daily['Clicks'].sum())}")
print(f"Total conversions: {df_daily['Conversions'].sum():.0f}")

## 4. Top Keywords
See which keywords are driving the most traffic and conversions.

In [3]:
# Top keywords by clicks (last 30 days)
query_keywords = """
    SELECT
        ad_group_criterion.keyword.text,
        ad_group_criterion.keyword.match_type,
        campaign.name,
        metrics.impressions,
        metrics.clicks,
        metrics.ctr,
        metrics.average_cpc,
        metrics.cost_micros,
        metrics.conversions
    FROM keyword_view
    WHERE segments.date DURING LAST_30_DAYS
        AND campaign.status != 'REMOVED'
    ORDER BY metrics.clicks DESC
    LIMIT 20
"""

response_kw = ga_service.search(customer_id=customer_id, query=query_keywords)

keywords = []
for row in response_kw:
    keywords.append({
        "Keyword": row.ad_group_criterion.keyword.text,
        "Match Type": row.ad_group_criterion.keyword.match_type.name,
        "Campaign": row.campaign.name,
        "Impressions": row.metrics.impressions,
        "Clicks": row.metrics.clicks,
        "CTR": f"{row.metrics.ctr:.2%}",
        "Avg CPC": f"${row.metrics.average_cpc / 1_000_000:.2f}",
        "Cost": f"${row.metrics.cost_micros / 1_000_000:.2f}",
        "Conversions": row.metrics.conversions,
    })

df_keywords = pd.DataFrame(keywords)
print(f"Top {len(df_keywords)} keywords by clicks\n")
df_keywords

Request made: ClientCustomerId: 8915520607, Host: googleads.googleapis.com, Method: /google.ads.googleads.v18.services.GoogleAdsService/Search, RequestId: None, IsFault: True, FaultMessage: GRPC target method can't be resolved.


MethodNotImplemented: 501 GRPC target method can't be resolved.

## 5. Search Terms Report
See what people actually searched for that triggered your ads.

In [4]:
# Search terms report (last 30 days)
query_search = """
    SELECT
        search_term_view.search_term,
        campaign.name,
        metrics.impressions,
        metrics.clicks,
        metrics.ctr,
        metrics.cost_micros,
        metrics.conversions
    FROM search_term_view
    WHERE segments.date DURING LAST_30_DAYS
        AND campaign.status != 'REMOVED'
    ORDER BY metrics.impressions DESC
    LIMIT 25
"""

response_st = ga_service.search(customer_id=customer_id, query=query_search)

search_terms = []
for row in response_st:
    search_terms.append({
        "Search Term": row.search_term_view.search_term,
        "Campaign": row.campaign.name,
        "Impressions": row.metrics.impressions,
        "Clicks": row.metrics.clicks,
        "CTR": f"{row.metrics.ctr:.2%}",
        "Cost": f"${row.metrics.cost_micros / 1_000_000:.2f}",
        "Conversions": row.metrics.conversions,
    })

df_search = pd.DataFrame(search_terms)
print(f"Top {len(df_search)} search terms by impressions\n")
df_search

Request made: ClientCustomerId: 8915520607, Host: googleads.googleapis.com, Method: /google.ads.googleads.v18.services.GoogleAdsService/Search, RequestId: None, IsFault: True, FaultMessage: GRPC target method can't be resolved.


MethodNotImplemented: 501 GRPC target method can't be resolved.